# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [2]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [3]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [5]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [6]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [7]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [8]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [9]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [10]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [11]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [12]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [13]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [14]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [15]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 17138.566, Val Loss: 12000.102


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 6121.729, Val Loss: 10942.304


In [16]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [17]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$139 $65 $28 $30 $37 $46 $37 $19 $26 $158 $10 $90 $46 $52 $26 $14 $20 $12 $54 $79 $33 $91 $3 $54 $178 $174 $228 $23 $39 $55 $51 $139 $67 $9 $67 $282 $28 $38 $106 $62 $155 $29 $28 $60 $0 $40 $25 $36 $53 $27 $27 $51 $113 $11 $104 $71 $27 $109 $7 $55 $114 $16 $41 $4 $310 $202 $98 $376 $19 $127 $17 $16 $114 $104 $8 $45 $103 $7 $14 $40 $65 $78 $61 $51 $10 $84 $175 $118 $6 $141 $22 $6 $2 $14 $44 $3 $5 $11 $27 $235 $3 $26 $12 $22 $7 $50 $61 $280 $1 $103 $42 $63 $102 $12 $22 $207 $19 $75 $17 $106 $14 $95 $64 $11 $99 $24 $15 $51 $65 $2 $98 $1 $80 $13 $79 $16 $101 $61 $5 $29 $11 $149 $30 $101 $52 $29 $15 $149 $68 $2 $14 $57 $4 $10 $24 $136 $65 $8 $14 $16 $56 $9 $1 $13 $289 $5 $62 $38 $20 $45 $50 $10 $315 $39 $1 $40 $47 $20 $2 $7 $114 $15 $212 $12 $16 $20 $81 $130 $51 $8 $19 $26 $6 $81 $30 $2 $175 $26 $15 $10 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [18]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [19]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [20]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [23]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openrouter/openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [24]:
gpt_4__1_nano(test[0])

'$250'

In [25]:
test[0].price

219.0

In [26]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $10 $45 $80 $6 $65 $11 $870 $163 $20 $25 $9 $19 $8 $41 $5 $40 $1 $59 $26 $35 $125 $212 $254 $455 $5 $251 $65 $30 $15 $10 $50 $35 $269 $60 $26 $6 $13 $165 $50 $25 $130 $70 $5 $32 $13 $75 $52 $20 $105 $225 $10 $97 $16 $8 $50 $52 $3 $86 $28 $46 $30 $571 $1 $90 $295 $25 $174 $17 $8 $30 $3 $25 $20 $26 $3 $8 $1 $30 $4 $5 $74 $7 $0 $68 $56 $0 $21 $3 $20 $5 $15 $2 $78 $1 $7 $70 $395 $20 $33 $17 $11 $50 $32 $12 $380 $19 $49 $10 $286 $99 $53 $34 $130 $1 $4 $34 $47 $24 $311 $80 $16 $50 $10 $10 $71 $59 $94 $79 $13 $65 $5 $85 $0 $55 $10 $78 $62 $16 $50 $10 $12 $119 $118 $15 $340 $15 $13 $3 $94 $17 $10 $3 $129 $101 $41 $20 $15 $311 $17 $6 $2 $140 $2 $752 $25 $10 $5 $5 $13 $20 $8 $22 $201 $3 $57 $105 $33 $246 $15 $150 $1 $25 $8 $63 $17 $20 $2 $5 $99 $5 $11 $50 $70 $10 $70 $21 $1 

In [29]:
def claude_opus_4_5(item):
    response = completion(model="openrouter/anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [30]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $34 $25 $25 $0 $70 $54 $25 $6 $31 $114 $129 $0 $21 $49 $3 $11 $20 $20 $74 $16 $1 $40 $25 $82 $253 $206 $4 $190 $64 $10 $30 $70 $50 $25 $289 $14 $43 $34 $8 $154 $55 $10 $45 $70 $0 $2 $2 $55 $112 $28 $114 $325 $10 $37 $44 $6 $35 $48 $1 $106 $48 $61 $60 $279 $9 $50 $355 $35 $14 $19 $3 $70 $6 $20 $11 $26 $2 $2 $6 $30 $3 $5 $74 $14 $25 $32 $44 $30 $16 $3 $5 $5 $22 $0 $99 $4 $67 $120 $325 $10 $27 $3 $49 $49 $28 $12 $365 $1 $114 $30 $36 $1 $38 $54 $29 $9 $7 $6 $47 $4 $161 $10 $76 $0 $15 $4 $9 $30 $93 $119 $12 $39 $0 $25 $2 $55 $10 $33 $11 $26 $249 $30 $7 $44 $2 $15 $25 $85 $8 $6 $83 $31 $94 $1 $89 $26 $43 $21 $20 $39 $18 $8 $0 $41 $7 $58 $20 $0 $3 $0 $9 $140 $14 $66 $1 $2 $3 $4 $43 $155 $10 $250 $59 $24 $3 $43 $17 $30 $14 $5 $1 $10 $11 $50 $10 $9 $130 $21 $11 

In [33]:
# Workaround: LiteLLM doesn't map reasoning_effort for OpenRouter; OpenRouter expects reasoning: { effort: "low" }.
# Merge user extra_body into OpenRouter request so extra_body={"reasoning": {"effort": "low"}} works.
import litellm
_orig_openrouter_map = litellm.OpenrouterConfig.map_openai_params
def _openrouter_map_with_extra_body(self, non_default_params, optional_params, model, drop_params):
    user_extra = non_default_params.pop("extra_body", {}) or {}
    result = _orig_openrouter_map(self, non_default_params, optional_params, model, drop_params)
    if user_extra and isinstance(result.get("extra_body"), dict):
        result["extra_body"].update(user_extra)
    return result
litellm.OpenrouterConfig.map_openai_params = _openrouter_map_with_extra_body

In [ ]:
def gemini_3_pro_preview(item):
    # OpenRouter uses reasoning: { effort: "low" } (see https://openrouter.ai/docs/use-cases/reasoning-tokens)
    # response = completion(model="openrouter/google/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    response = completion(
        model="openrouter/google/gemini-3-pro-preview",
        messages=messages_for(item),
        extra_body={"reasoning": {"effort": "low"}},
        allowed_openai_params=["extra_body"],
    )
    return response.choices[0].message.content

In [35]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]

$0 $64 $5 $25 $0 $140 $94 $95 $15 $170 $137 $130 $5 $9 $49 $8 $11 $19 $30 $34 $14 $9 $30 $15 $42 $204 $245 $7 $111 $60 $1 $35 $39 $55 $35 $219 $5 $41 $14 $18 $125 $40 $11 $105 $30 $5 $1 $2 $90 $2 

In [38]:
def gemini_2__5_flash_lite(item):
    response = completion(model="openrouter/google/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [39]:
evaluate(gemini_2__5_flash_lite, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$41 $134 $5 $80 $45 $15 $79 $65 $9 $20 $363 $30 $25 $9 $1 $2 $120 $5 $70 $69 $34 $4 $10 $45 $162 $203 $145 $5 $151 $60 $15 $0 $10 $50 $5 $19 $15 $31 $64 $13 $140 $25 $10 $45 $50 $0 $2 $3 $75 $52 $27 $95 $225 $10 $7 $16 $8 $30 $27 $7 $86 $13 $56 $25 $29 $30 $20 $295 $25 $74 $17 $8 $130 $4 $15 $11 $51 $0 $8 $4 $30 $1 $5 $74 $8 $190 $168 $56 $0 $14 $3 $55 $15 $10 $2 $48 $11 $37 $120 $225 $10 $17 $8 $39 $26 $67 $13 $350 $4 $0 $5 $436 $24 $53 $54 $110 $1 $4 $12 $47 $9 $211 $55 $14 $10 $5 $15 $51 $41 $74 $29 $8 $15 $0 $85 $10 $85 $20 $53 $12 $9 $550 $50 $6 $81 $43 $15 $140 $285 $8 $1 $94 $12 $10 $6 $71 $16 $41 $30 $25 $89 $11 $78 $2 $190 $12 $1052 $14 $35 $5 $0 $3 $220 $8 $27 $280 $6 $13 $106 $18 $4 $15 $250 $1 $15 $8 $73 $12 $10 $2 $45 $9 $10 $161 $25 $70 $29 $0 $6 $1 

In [44]:
def grok_4__1_fast(item):
    response = completion(
        model="openrouter/x-ai/grok-4.1-fast",
        messages=messages_for(item),
        seed=42,
        extra_body={"reasoning": {"enabled": False}},
        allowed_openai_params=["extra_body"],
    )
    return response.choices[0].message.content

In [45]:
evaluate(grok_4__1_fast, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $34 $30 $59 $20 $110 $54 $95 $11 $80 $87 $120 $5 $4 $19 $8 $1 $5 $90 $69 $16 $26 $35 $175 $18 $153 $45 $5 $301 $65 $30 $20 $90 $65 $14 $419 $60 $31 $14 $13 $175 $45 $25 $105 $100 $0 $7 $3 $65 $48 $25 $117 $225 $20 $3 $16 $3 $40 $48 $0 $116 $48 $46 $60 $179 $10 $10 $325 $5 $174 $17 $8 $30 $4 $30 $16 $26 $0 $4 $6 $40 $1 $5 $74 $2 $20 $68 $56 $30 $21 $3 $5 $5 $20 $4 $108 $4 $37 $20 $325 $20 $3 $12 $10 $99 $132 $17 $350 $9 $49 $10 $64 $19 $48 $54 $230 $7 $0 $34 $147 $9 $411 $10 $16 $20 $10 $5 $51 $29 $49 $179 $13 $5 $5 $185 $0 $55 $10 $22 $18 $16 $349 $40 $17 $44 $18 $5 $290 $185 $8 $4 $144 $22 $60 $4 $29 $71 $41 $100 $5 $211 $13 $78 $2 $140 $2 $402 $25 $5 $5 $10 $7 $120 $8 $57 $1 $7 $27 $14 $27 $146 $15 $150 $1 $30 $3 $83 $17 $10 $2 $5 $1 $10 $111 $11 $50 $80 $30 $26 $1 

In [48]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="openrouter/openai/gpt-5.1", messages=messages_for(item), 
    extra_body={"reasoning": {"effort": "high"}},
    seed=42)
    return response.choices[0].message.content


In [49]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $84 $5 $10 $20 $150 $54 $95 $9 $50 $237 $79 $0 $7 $44 $3 $21 $20 $20 $39 $26 $6 $5 $25 $53 $223 $165 $3 $91 $64 $9 $30 $30 $50 $5 $120 $40 $37 $44 $15 $150 $40 $13 $65 $40 $0 $5 $1 $75 $78 $26 $110 $150 $0 $37 $39 $4 $49 $58 $3 $116 $48 $36 $65 $129 $0 $50 $315 $5 $44 $17 $2 $100 $0 $22 $18 $6 $1 $1 $6 $0 $3 $10 $74 $12 $25 $58 $76 $0 $18 $3 $15 $5 $5 $0 $88 $4 $87 $90 $245 $10 $7 $2 $9 $19 $82 $15 $355 $6 $130 $20 $85 $9 $58 $54 $0 $6 $6 $14 $496 $9 $61 $10 $16 $10 $15 $0 $21 $19 $59 $148 $27 $5 $0 $45 $3 $55 $10 $92 $2 $16 $349 $10 $8 $14 $4 $5 $60 $5 $8 $4 $104 $25 $60 $4 $121 $31 $43 $60 $15 $10 $17 $7 $2 $641 $1 $451 $25 $0 $1 $10 $2 $170 $13 $42 $1 $6 $37 $14 $8 $124 $15 $230 $49 $20 $6 $73 $12 $10 $2 $5 $19 $10 $81 $60 $50 $10 $0 $21 $9 